# 01 · Download the QKI shRNA RNA-seq BAMs (panel c inputs)

Second of six notebooks (00 screen and locus → **01 download** → 02 rMATS → 03 panel c → 04 eCLIP and rbp-maps → 05 optional window maps).

**Which BAMs.** Panel c uses ENCODE's rMATS skipped-exon results for the two QKI knockdowns. Each rMATS file records the BAMs it was computed from (its `derived_from` field): two shRNA replicates and two non-target shRNA control replicates. They are TopHat alignments to **hg19** of 100-nt paired-end reads, quantified against GENCODE v19. This notebook reads that lineage from the ENCODE API, checks it against the table below, and downloads the files.

| Knockdown | rMATS | Condition | Rep | Experiment | BAM | Size |
| --- | --- | --- | --- | --- | --- | --- |
| QKI, K562 | ENCFF096LJD | QKI shRNA | 1 | ENCSR256PLH | ENCFF280ZIB | 4.0 GB |
| | | QKI shRNA | 2 | ENCSR256PLH | ENCFF708KBX | 3.9 GB |
| | | non-target control | 1 | ENCSR344XID | ENCFF092XAP | 3.9 GB |
| | | non-target control | 2 | ENCSR344XID | ENCFF265ZZB | 4.5 GB |
| QKI, HepG2 | ENCFF103PLX | QKI shRNA | 1 | ENCSR330YOU | ENCFF610CID | 5.7 GB |
| | | QKI shRNA | 2 | ENCSR330YOU | ENCFF262GAC | 5.6 GB |
| | | non-target control | 1 | ENCSR603TCV | ENCFF755GKJ | 6.3 GB |
| | | non-target control | 2 | ENCSR603TCV | ENCFF571ARP | 4.8 GB |

Total: **38.7 GB** of BAMs. Downloads resume if interrupted; every file is checked against ENCODE's MD5.

**Environment.** Use the `encode-rbp-case-study` conda environment (`notebooks/environment.yml`); this notebook needs `requests`, `pandas` and `samtools`.

## Accessing RBP-ARK data through Globus

RBP-ARK keeps every Skipper output on one Globus collection. Public files can be downloaded straight over HTTPS with no login; bulk copies go through Globus.

| | |
| --- | --- |
| Globus collection | **RBP-ARK**, `9cd9be17-cbed-4b34-ab76-704b2fd6ac7b` ([open in the Globus web app](https://app.globus.org/file-manager?origin_id=9cd9be17-cbed-4b34-ab76-704b2fd6ac7b&origin_path=%2F)) |
| HTTPS address of the same files | `https://g-f8db23.0ed28.75bc.data.globus.org/<path>` |

**Where things are.** `<version>/<run>` is the Skipper version and run folder. Since November 2026 every ENCODE3, ENCORE and ENCORE2 dataset on rbp-ark.com (gene search, dataset pages, BDBags) is Skipper `2.0.0/v2`, and so are these notebooks. Earlier runs (`1.99.0/v1`, `1.100.0/v1`) remain on the collection.

| Path on the collection | Contents |
| --- | --- |
| `/encode3/eclip/Skipper/<version>/<run>/output/` | ENCODE3 eCLIP datasets, named `<RBP>_<cell>_ENCSR…` |
| `/encore2/eclip/Skipper/<version>/<run>/output/` | ENCORE2 eCLIP datasets with 5000-series IDs (e.g. `QKI_K562_5007`) |
| `/encore/eclip/Skipper/<version>/<run>/output/` | ENCORE eCLIP datasets with 4000-series IDs |
| `/summary_data/` | summary tables (QC, window counts) |

Inside each `output/` folder, files are grouped by type, then named by dataset:

| Subfolder | Files |
| --- | --- |
| `reproducible_enriched_windows/` | `<dataset>.reproducible_enriched_windows.tsv.gz`, the enriched windows |
| `reproducible_enriched_re/` | enriched repetitive elements |
| `secondary_results/bams/dedup/genome_R2/` | deduplicated, read-2-only genome BAMs (ENCODE3) |
| `secondary_results/bams/dedup/genome/` | deduplicated genome BAMs (ENCORE) |
| `secondary_results/bams/raw/genome/` | alignments before deduplication |
| `secondary_results/bigwigs/scaled/plus/`, `.../minus/` | strand-specific signal tracks |
| `QC/`, `gene_sets/`, `homer/` | QC reports, gene set enrichment, motifs |

**Three ways to get files.**

1. **One dataset: BDBag.** Each dataset page on rbp-ark.com has a **Download BDBag** link. The bag's `fetch.txt` lists every file of that dataset with its HTTPS URL, size and path, and `manifest-sha256.txt` its checksum. Notebook 04 reads the same file list from the RBP-ARK API and downloads only the BAMs it needs.
2. **Many datasets: Globus web app.** Sign in at [app.globus.org](https://app.globus.org) with your institution or a free Globus ID, open the RBP-ARK collection (link above), select folders or files, and transfer them to a collection you can write to: your laptop via [Globus Connect Personal](https://www.globus.org/globus-connect-personal), or your institution's cluster or storage collection.
3. **Scripted: Globus CLI.**

```bash
pip install globus-cli
globus login                                   # once; opens a browser
RBPARK=9cd9be17-cbed-4b34-ab76-704b2fd6ac7b
OUT=/encode3/eclip/Skipper/2.0.0/v2/output

# list what is there
globus ls "$RBPARK:$OUT/secondary_results/bams/dedup/genome_R2/" | grep QKI

# copy one file to this computer (Globus Connect Personal must be running)
DEST=$(globus endpoint local-id)
globus transfer "$RBPARK:$OUT/secondary_results/bams/dedup/genome_R2/QKI_K562_ENCSR366YOG_IP_1.genome.Aligned.sort.dedup.R2.bam" \
    "$DEST:$HOME/rbpark/QKI_K562_ENCSR366YOG_IP_1.genome.Aligned.sort.dedup.R2.bam" --label "QKI eCLIP"

# or copy a whole folder
globus transfer --recursive "$RBPARK:$OUT/reproducible_enriched_windows/" "$DEST:$HOME/rbpark/windows/"
globus task list                               # follow progress
```

If the CLI asks for extra consent the first time it touches the collection, it prints the exact `globus session consent ...` command to run. The Globus CLI commands are shown for reference and were not run in testing; the BDBag and HTTPS routes were. Please copy only what you need: the public HTTPS endpoint is shared, and bulk sweeps have taken it offline.

**In this notebook:** nothing comes from RBP-ARK; the knockdown BAMs are ENCODE files. RBP-ARK data is used in notebooks 00, 04 and 05.

In [ ]:
import hashlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd
import requests

sys.path.insert(0, ".")
import cs3_utils as U

ROOT, DATA = U.folders()
BAM_DIR = DATA / "downloads" / "shrna_bams_hg19"
REF_DIR = DATA / "downloads" / "encode_reference"
for d in (BAM_DIR, REF_DIR):
    d.mkdir(parents=True, exist_ok=True)

ENCODE = "https://www.encodeproject.org"
VERIFY_MD5 = True  # about 1 min per GB; set False to skip
THREADS = os.cpu_count() or 4

assert shutil.which("samtools"), "samtools is not on PATH (activate the encode-rbp-case-study environment)"
print("ROOT:", ROOT, "| DATA:", DATA)
pd.DataFrame(U.KNOCKDOWNS).T

## 1. Read the BAM lineage from the ENCODE API

`EXPECTED` is the table above. The cell below asks ENCODE which BAMs each rMATS file was derived from and stops if the answer differs.

In [ ]:
EXPECTED = pd.DataFrame([
    ("QKI_K562", "knockdown", 1, "ENCSR256PLH", "ENCFF280ZIB"), ("QKI_K562", "knockdown", 2, "ENCSR256PLH", "ENCFF708KBX"),
    ("QKI_K562", "control", 1, "ENCSR344XID", "ENCFF092XAP"), ("QKI_K562", "control", 2, "ENCSR344XID", "ENCFF265ZZB"),
    ("QKI_HepG2", "knockdown", 1, "ENCSR330YOU", "ENCFF610CID"), ("QKI_HepG2", "knockdown", 2, "ENCSR330YOU", "ENCFF262GAC"),
    ("QKI_HepG2", "control", 1, "ENCSR603TCV", "ENCFF755GKJ"), ("QKI_HepG2", "control", 2, "ENCSR603TCV", "ENCFF571ARP"),
], columns=["knockdown", "condition", "rep", "experiment", "accession"])


def encode_json(path):
    """Returns the ENCODE object at `path` (for example /files/ENCFF096LJD/) as JSON."""
    r = requests.get(ENCODE + path, params={"format": "json"},
                     headers={"accept": "application/json"}, timeout=120)
    r.raise_for_status()
    return r.json()


def object_id(x):
    """Returns the @id of a derived_from entry `x`, which is either an '@id' string or an embedded object."""
    return x if isinstance(x, str) else x["@id"]


rows = []
for key, kd in U.KNOCKDOWNS.items():
    rmats = encode_json("/files/{}/".format(kd["rmats"]))
    print(key, kd["rmats"], "|", rmats.get("assembly"), "| annotation", rmats.get("genome_annotation"))
    for parent in rmats["derived_from"]:
        bam = encode_json(object_id(parent))
        experiment = object_id(bam["dataset"]).strip("/").split("/")[-1]
        rows.append({
            "knockdown": key, "rbp": kd["rbp"], "cell": kd["cell"],
            "condition": "knockdown" if experiment == kd["kd_experiment"] else "control",
            "rep": bam["biological_replicates"][0],
            "experiment": experiment,
            "accession": bam["accession"],
            "assembly": bam["assembly"],
            "run_type": bam.get("mapped_run_type"),
            "read_length": bam.get("mapped_read_length"),
            "file_size": bam["file_size"],
            "md5sum": bam["md5sum"],
            "href": bam["href"],
        })

lineage = pd.DataFrame(rows).sort_values(["knockdown", "condition", "rep"], ascending=[True, False, True])
check = lineage.merge(EXPECTED, on=["knockdown", "condition", "rep", "experiment", "accession"], how="outer", indicator=True)
assert (check["_merge"] == "both").all(), check
lineage[["knockdown", "condition", "rep", "experiment", "accession", "assembly", "run_type", "read_length", "file_size"]]

## 2. Download and verify

`curl -C -` resumes a partial file. A file whose size and MD5 already match is skipped, so the cell is safe to re-run.

In [ ]:
have = sum(Path(BAM_DIR / (a + ".bam")).stat().st_size for a in lineage["accession"] if (BAM_DIR / (a + ".bam")).exists())
need = int(lineage["file_size"].sum()) - have
free = shutil.disk_usage(BAM_DIR).free
print("still to download {:.1f} GB, free {:.1f} GB on {}".format(need / 1e9, free / 1e9, BAM_DIR))
assert free > need * 1.05, "not enough disk space for the BAMs"


def md5(path, chunk=8 << 20):
    """Returns the MD5 checksum of the file at `path`, read `chunk` bytes at a time."""
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def download(url, out, size, md5sum=None):
    """Downloads `url` to `out` (resuming a partial file) and returns `out`; checks that it is `size` bytes and, if
    given, matches `md5sum`. A verified file is recorded (.md5ok) and not checked again.
    """
    ok = Path(str(out) + ".md5ok")
    if ok.exists() and out.exists() and out.stat().st_size == size:
        return out
    if not (out.exists() and out.stat().st_size == size):
        subprocess.check_call(["curl", "-L", "-sS", "--retry", "5", "-C", "-", "-o", str(out), url])
    assert out.stat().st_size == size, "{}: size {} != {}".format(out.name, out.stat().st_size, size)
    if VERIFY_MD5 and md5sum:
        assert md5(out) == md5sum, "{}: MD5 mismatch; delete it and re-run".format(out.name)
        ok.write_text(md5sum + "\n")
    return out


paths = []
for _, row in lineage.iterrows():
    out = BAM_DIR / "{}.bam".format(row["accession"])
    print("downloading", row["knockdown"], row["condition"], "rep", row["rep"], "->", out.name, flush=True)
    paths.append(download(ENCODE + row["href"], out, row["file_size"], row["md5sum"]))
lineage["bam"] = [str(p) for p in paths]

## 3. Index and sanity-check

rMATS does not need an index, but rmats2sashimiplot (notebook 03) does. The last column counts reads over PICALM exons 12-14 (hg19 `chr11:85694908-85707972`), the region panel c shows.

In [ ]:
PICALM_REGION = "chr11:85694908-85707972"  # hg19, the exons flanking exon 13 (minus strand)

summary = []
for bam in lineage["bam"]:
    if not Path(bam + ".bai").exists():
        subprocess.check_call(["samtools", "index", "-@", str(THREADS), bam])
    idx = subprocess.check_output(["samtools", "idxstats", bam], text=True)
    mapped = sum(int(line.split("\t")[2]) for line in idx.strip().splitlines())
    picalm = int(subprocess.check_output(["samtools", "view", "-c", "-F", "260", bam, PICALM_REGION], text=True))
    summary.append({"bam": bam, "mapped_reads": mapped, "picalm_exons12_14_reads": picalm})

lineage = lineage.drop(columns=["mapped_reads", "picalm_exons12_14_reads"], errors="ignore").merge(pd.DataFrame(summary), on="bam")
lineage[["knockdown", "condition", "rep", "accession", "mapped_reads", "picalm_exons12_14_reads"]]

## 4. ENCODE's own outputs, for comparison

- The released rMATS results (`MATS_output`, rMATS v3.2.1.beta). Notebook 02 compares its rerun to these, and notebooks 03 and 04 can use them directly (`SE_SOURCE=encode`).
- Two DESeq2 tables per knockdown: without batch normalization (used here for the knockdown level: QKI log2 fold change −0.85 in K562 and −3.36 in HepG2) and batch-normalized (QKI −1.21 and −6.06). Notebook 03 prints both.

In [ ]:
reference = []
for key, kd in U.KNOCKDOWNS.items():
    for role in ("rmats", "deseq2", "deseq2_batch"):
        acc = kd[role]
        meta = encode_json("/files/{}/".format(acc))
        out = REF_DIR / Path(meta["href"]).name
        download(ENCODE + meta["href"], out, meta["file_size"], meta["md5sum"])
        reference.append({"knockdown": key, "role": role, "accession": acc, "file": out.name,
                          "alias": ";".join(meta.get("aliases", []))})
pd.DataFrame(reference)

In [ ]:
# Manifest read by notebooks 02-03
manifest = lineage[["knockdown", "rbp", "cell", "condition", "rep", "experiment", "accession", "bam", "mapped_reads"]]
manifest.to_csv(BAM_DIR / "manifest.tsv", sep="\t", index=False)
print("wrote", BAM_DIR / "manifest.tsv")
manifest